# Getting Started with NeMo Agent Toolkit

In this notebook, we walk through the basics of using NVIDIA NeMo Agent toolkit (NAT), from installation all the way to creating and running a simple workflow. The intention of this notebook is to get new NAT users up and running with a high level understanding of our YAML-first approach, while gaining some intuition towards how NAT workflows can quickly be embedded into your projects.

## Table of Contents

- [0) Setup](#setup)
  - [0.1) Prerequisites](#prereqs)
  - [0.2) API Keys](#api-keys)
  - [0.3) Installing NeMo Agent Toolkit](#installing-nat)
- [1) Creating Your First Workflow](#creating-your-first-workflow)
  - [1.1) What is a NAT workflow?](#what-is-a-workflow)
  - [1.2) Create your first workflow](#create-first-workflow)
  - [1.3) Interpret your first workflow](#interpret-first-workflow)
    - [Interpreting Directory Structure](#directory-structure)
    - [Interpreting Configuration File](#configuration-file)
    - [Interpreting Workflow Functions](#workflow-functions)
    - [Tying It Together](#tying-it-together)
- [2) Running Your First Workflow](#run-first-workflow)
    - [2.1) Run with the CLI](#run-cli)
    - [2.2) Run as a NAT server](#run-server)
    - [2.3) Running NAT Embedded within Python](#run-embedded)
- [Next Steps](#next-steps)

<span style="color:rgb(0, 31, 153); font-style: italic;">Note: In Google Colab use the Table of Contents tab to navigate.</span>



<a id="setup"></a>
# 0.0) Setup

<a id="prereqs"></a>
## 0.1) Prerequisites

- **Platform:** Linux, macOS, or Windows
- **Python:** version 3.11, 3.12, or 3.13
- **Python Packages:** `pip`

<a id="api-keys"></a>
## 0.2) API Keys

For this notebook, you will need the following API keys to run all examples end-to-end:

- **NVIDIA Build:** You can obtain an NVIDIA Build API Key by creating an [NVIDIA Build](https://build.nvidia.com) account and generating a key at https://build.nvidia.com/settings/api-keys

Then you can run the cell below:

In [5]:
import getpass
import os

if "NVIDIA_API_KEY" not in os.environ:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA API key: ")
    os.environ["NVIDIA_API_KEY"] = nvidia_api_key

<a id="installing-nat"></a>
## 0.3) Installing NeMo Agent Toolkit

The recommended way to install NAT is through `pip` or `uv pip`.

First, we will install `uv` which offers parallel downloads and faster dependency resolution.

In [ ]:
!pip install uv

NeMo Agent toolkit can be installed through the PyPI `nvidia-nat` package.

There are several optional subpackages available for NAT. The `langchain` subpackage contains useful components for integrating and running within [LangChain](https://python.langchain.com/docs/introduction/). Since LangChain will be used later in this notebook, let's install NAT with the optional `langchain` subpackage.

In [ ]:
%%bash
uv pip show -q "nvidia-nat-langchain"
if [ $? -ne 0 ]; then
    ## build from pypi
    # uv pip install "nvidia-nat[langchain]"

    ## build from source
    cd ../../
    uv pip install -e ".[langchain]"
else
    echo "nvidia-nat[langchain] is already installed"
fi

<a id="creating-your-first-workflow"></a>
# 1.0) Creating Your First Workflow

<a id="what-is-a-workflow"></a>
## 1.1) What is a NAT workflow?

A [workflow](https://docs.nvidia.com/nemo/agent-toolkit/latest/workflows/about/index.html) in NeMo Agent Toolkit is a structured specification of how agents, models, tools (called functions), embedders, and other components are composed together to carry out a specific task. It defines which components are used, how they are connected, and how they behave when executing the task.

NAT provides a convenient command-line interface called `nat` which is accessible in your active Python environment. It serves at the entrypoint to most toolkit functions.

The `nat workflow create` command allows us to create a new workflow.

<a id="create-first-workflow"></a>
## 1.2) Create your first workflow

In [ ]:
!nat workflow create getting_started

If you are building nat **from source**, you may encouter error "configuration error: `project.dependencies[0]` must be pep508". It's because the local build version cannot match with `~=` and violate pep508 standard. 

Modify dependencies in `pyproject.toml` as below:

```
dependencies = [
  "nvidia-nat[langchain]",
]
```

Then rebuild project:


In [ ]:
!uv pip install -e "getting_started/."

<a id="interpret-first-workflow"></a>
## 1.3) Interpret your first workflow

<a id="directory-structure"></a>
### Interpreting Directory Structure
We can inspect the structure of the created **workflow directory**, which we've named `getting_started`, and contains the configuration files, source code, and data needed to define and run the workflow.

In [ ]:
!find getting_started/

A summary of the high-level components are outlined below.

* `configs` (symbolic link to `src/getting_started/configs`)
* `data` (symbolic link to `src/getting_started/data`)
* `pyproject.toml` Python project configuration file
* `src`
  * `getting_started`
    * `__init__.py` Module init file (empty)
    * `configs` Configuration directory for workflow specifications
      * `config.yml` Workflow configuration file
    * `data` Data directory for any dependent files
    * `getting_started.py` User-defined code for workflow execution
    * `register.py` Automatic registration of project components


<a id="configuration-file"></a>
### Interpreting Configuration File
The workflow configuration file, `getting_started/configs/config.yml`, describes the operational characteristics of the entire workflow. Let's load its contents in the next cell and understand what this first workflow can do out of the box.

In [ ]:
# %load getting_started/configs/config.yml
functions:
  current_datetime:
    _type: current_datetime
  getting_started:
    _type: getting_started
    prefix: "Hello:"

llms:
  nim_llm:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
    temperature: 0.0

workflow:
  _type: react_agent
  llm_name: nim_llm
  tool_names: [current_datetime, getting_started]

The above workflow configuration has the following components:
- a [built-in `current_datetime`](https://docs.nvidia.com/nemo/agent-toolkit/latest/api/nat/tool/datetime_tools/index.html#nat.tool.datetime_tools.current_datetime) function
- a workflow-defined `getting_started` function
- an LLM
- an entrypoint workflow of a [built-in ReAct agent](https://docs.nvidia.com/nemo/agent-toolkit/latest/workflows/about/react-agent.html)

By default, we create a [ReAct agent](https://docs.nvidia.com/nemo/agent-toolkit/latest/workflows/about/react-agent.html) equipped with both of the functions above. When called, the Agent decides which functions to call (if any) based on the intent of user input. The agent uses the LLM to help make reasoning decisions and then performs a subsequent action.

This workflow configuration file is a YAML-serialized version of the [`Config`](https://docs.nvidia.com/nemo/agent-toolkit/latest/api/nat/data_models/config/index.html#nat.data_models.config.Config) class. Each category within the high-level configuration specifies runtime configuration settings for their corresponding components. For instance, the `workflow` category contains all configuration settings for the workflow entrypoint. This configuration file is validated as typed Pydantic models and fields. All configuration classes have validation rules, default values, and [documentation](https://docs.nvidia.com/nemo/agent-toolkit/latest/workflows/workflow-configuration.html#workflow-configuration-file) which enable type-safe configuration management, automatic schema generation, and validation across the entire plugin ecosystem.

* `general` - General configuration section. Contains high-level configurations for front-end definitions.
* `authentication` - Authentication provides an interface for defining and interacting with various authentication providers.
* `llms` - LLMs provide an interface for interacting with LLM providers.
* `embedders` - Embedders provide an interface for interacting with embedding model providers.
* `retreivers` - Retrievers provide an interface for searching and retrieving documents.
* `memory` - Configurations for Memory. Memories provide an interface for storing and retrieving.
* `object_stores` - Object Stores provide a CRUD interface for objects and data.
* `eval` - The evaluation section provides configuration options related to the profiling and evaluation of NAT workflows.
* `tcc_strategies` (experimental) - Test Time Compute (TTC) strategy definitions.

#### Type Safety and Validation

Many components within the workflow configuration specify `_type`. This YAML key is used to indicate the type of the component so NAT can properly validate and instantiate a component within the workflow. For example, [`NIMModelConfig`](https://docs.nvidia.com/nemo/agent-toolkit/latest/api/nat/llm/nim_llm/index.html#nat.llm.nim_llm.NIMModelConfig) is a subclass of [`LLMBaseConfig`](https://docs.nvidia.com/nemo/agent-toolkit/latest/api/nat/data_models/llm/index.html#nat.data_models.llm.LLMBaseConfig) so when we specify: `_type: nim` in the configuration the toolkit knows to validate the configuration with `NIMModelConfig`.

<span style="color:rgb(0, 31, 153); font-style: italic;">**Note:** Not all configuration components are required. The simplest workflow configuration needs to only define <code>workflow</code>.</span>




<a id="workflow-functions"></a>
## 1.4) Interpreting Workflow Functions

Next, let's inspect the contents of the generated workflow function:

In [ ]:
# %load getting_started/src/getting_started/getting_started.py
import logging

from pydantic import Field

from nat.builder.builder import Builder
from nat.builder.framework_enum import LLMFrameworkEnum
from nat.builder.function_info import FunctionInfo
from nat.cli.register_workflow import register_function
from nat.data_models.function import FunctionBaseConfig

logger = logging.getLogger(__name__)


class GettingStartedFunctionConfig(FunctionBaseConfig, name="getting_started"):
    """
    NAT function template. Please update the description.
    """
    prefix: str = Field(default="Echo:", description="Prefix to add before the echoed text.")


@register_function(config_type=GettingStartedFunctionConfig, framework_wrappers=[LLMFrameworkEnum.LANGCHAIN])
async def getting_started_function(config: GettingStartedFunctionConfig, builder: Builder):
    """
    Registers a function (addressable via `getting_started` in the configuration).
    This registration ensures a static mapping of the function type, `getting_started`, to the `GettingStartedFunctionConfig` configuration object.

    Args:
        config (GettingStartedFunctionConfig): The configuration for the function.
        builder (Builder): The builder object.

    Returns:
        FunctionInfo: The function info object for the function.
    """

    # Define the function that will be registered.
    async def _echo(text: str) -> str:
        """
        Takes a text input and echoes back with a pre-defined prefix.

        Args:
            text (str): The text to echo back.

        Returns:
            str: The text with the prefix.
        """
        return f"{config.prefix} {text}"

    # The callable is wrapped in a FunctionInfo object.
    # The description parameter is used to describe the function.
    yield FunctionInfo.from_fn(_echo, description=_echo.__doc__)

### Function Configuration

The `GettingStartedFunctionConfig` specifies `FunctionBaseConfig` as a base class. There is also a `name` specified. This name is used by the toolkit to create a static mapping when `_type` is specified anywhere where a `FunctionBaseConfig` is expected, such as `workflow` or under `functions`.

### Function Registration

NeMo Agent toolkit relies on a configuration with builder pattern to define most components. For functions, `@register_function` is a decorator that must be specified to inform the toolkit that a function should be accessible automatically by name when referenced. The decorator requires that a `config_type` is specified. This is done to ensure type safety and validation.

The parameters to the decorated function are always:

1. the configuration type of the function component (FunctionBaseConfig)
2. a Builder which can be used to dynamically query and get other workflow components (Builder)

### Function Implementation

The core logic of the `getting_started` function is embedded as a function within the outer function registration. This is done for a few reasons:

* Enables dynamic importing of libraries and modules on an as-needed basis.
* Enables context manager-like resources within to support automatic closing of resources.
* Provides the most flexibility to users when defining their own functions.

Near the end of the function registration implementation, we `yield` a `FunctionInfo` object. `FunctionInfo` is a wrapper around any type of function. It is also possible to specify additional information such as schema and converters if your function relies on transformations.

NAT relies on `yield` rather `return` so resources can stay alive during the lifetime of the function or workflow.

<a id="tying-it-together"></a>
### Tying It Together

Looking back at the configuration file, the `workflow`'s `_type` is `getting_started`. This means that the configuration of `workflow` will be validated based on the `GettingStartedFunctionConfig` implementation.

The `register.py` file tells NAT what should automatically be imported so it is available when the toolkit is loaded.

In [ ]:
# %load getting_started/src/getting_started/register.py
# flake8: noqa

# Import the generated workflow function to trigger registration
from .getting_started import getting_started_function

<a id="run-first-workflow"></a>
# 2.0) Running Your First Workflow

<a id="run-cli"></a>
## 2.1) Run with the CLI

You can run a workflow by using `nat run` CLI command:

In [1]:
!nat run --config_file getting_started/configs/config.yml \
         --input "What is the current time?"

2025-12-18 01:23:33 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'getting_started/configs/config.yml'

Configuration Summary:
--------------------
Workflow Type: react_agent
Number of Functions: 2
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2025-12-18 01:23:42 - INFO     - nat.front_ends.console.console_front_end_plugin:102 - --------------------------------------------------
Workflow Result:
['The current time is 2025-12-18 01:23:34 +0000']
--------------------------------------------------


运行 `!nat run --config_file getting_started/configs/config.yml --input "What is the current time?"` 时，NeMo Agent Toolkit (NAT) 实际上经历了一个从 **配置加载** -> **组件发现与注册** -> **实例化构建** -> **运行时执行** 的完整生命周期。


### 第一阶段：CLI 入口与配置解析 (Bootstrap)

1.  **命令解析**：
    `nat run` 是 `nat start console` 的别名。程序启动后，首先由 CLI 解析器读取命令行参数，定位到 `--config_file` 指定的 YAML 文件。

2.  **加载 YAML 配置**：
    NAT 读取 `getting_started/configs/config.yml`。这个文件是整个 Workflow 的**蓝图**。
    *   它定义了**组件清单**：需要什么函数 (`functions`)、什么模型 (`llms`)。
    *   它定义了**组装方式**：`workflow` 部分指定了入口类型是 `react_agent`（推理-行动代理），并告诉代理可以使用哪些工具 (`tool_names`)。

### 第二阶段：组件发现与注册 (Discovery & Registration)

这是最关键的环节，解释了为什么 YAML 里的 `_type: getting_started` 能找到你写的 Python 代码。

1.  **Entry Points 扫描**：
    NAT 启动时会扫描当前 Python 环境中所有安装包的 `entry_points`。
    在你的 `getting_started/pyproject.toml` 文件中，定义了这样一个入口点：
    ```toml
    [project.entry-points.'nat.components']
    getting_started = "getting_started.register"
    ```
    这告诉 NAT：“如果看到有人在其配置中请求 `getting_started` 组件，请去加载 `getting_started.register` 模块。”

2.  **模块导入与装饰器执行**：
    *   NAT 加载 `src/getting_started/register.py`。
    *   `register.py` 导入了 `src/getting_started/getting_started.py`。
    *   在 `getting_started.py` 中，Python解释器执行到 `@register_function` 装饰器：
        ```python
        @register_function(config_type=GettingStartedFunctionConfig, framework_wrappers=[LLMFrameworkEnum.LANGCHAIN])
        async def getting_started_function(...)
        ```
    *   **注册动作**：此时，字符串 `"getting_started"`（来自 Config 类的 `name`）被正式注册到 NAT 的内部注册表中，并映射到 `GettingStartedFunctionConfig` 配置类和构建函数 `getting_started_function`。

### 第三阶段：实例化与构建 (Builder Phase)

现在 NAT 知道了所有的组件定义，开始根据 YAML 实例化对象。

1.  **配置验证 (Pydantic)**：
    NAT 看到 YAML 中定义的：
    ```yaml
    getting_started:
      _type: getting_started
      prefix: "Hello:"
    ```
    它会使用之前注册的 `GettingStartedFunctionConfig` 类来验证这段 YAML。它检查 `prefix` 是否是字符串，是否符合默认值等。

2.  **执行构建函数**：
    验证通过后，NAT 调用 `getting_started.py` 中的 `getting_started_function(config, builder)`。
    *   **Config 注入**：YAML 中的参数（如 `prefix="Hello:"`）被注入到 `config` 对象中。
    *   **闭包创建**：函数内部定义了 `async def _echo(text: str)`。注意，这个内部函数可以直接访问外部的 `config.prefix`。
    *   **Yield 组件**：函数最后 `yield FunctionInfo(...)`，将封装好的、可执行的 `_echo` 函数返回给 NAT 系统。

3.  **组装 Agent**：
    NAT 初始化 `workflow` 部分定义的 `react_agent`。
    *   它加载 `nim_llm`（NVIDIA 的云端大模型）。
    *   它将实例化好的工具列表（`current_datetime` 和刚刚构建的 `getting_started`）挂载到 Agent 上。

### 第四阶段：运行时执行 (Runtime Execution)

一切准备就绪，程序进入执行循环。

1.  **接收输入**：
    CLI 将用户输入 `"What is the current time?"` 传递给 Agent。

2.  **ReAct 循环 (Reasoning & Acting)**：
    *   **思考 (Think)**：Agent 将用户问题 + 工具描述（由 `FunctionInfo` 提供）打包成 Prompt 发送给 LLM。
    *   **决策**：LLM 分析后认为需要调用工具（例如 `current_datetime`），返回一个特定的 Token 序列。
    *   **行动 (Act)**：NAT 捕获到 LLM 的工具调用请求，执行对应的 Python 函数。
    *   **观察 (Observe)**：函数执行结果返回给 Agent。
    *   **再次思考**：Agent 将“观察结果”再次发给 LLM。

3.  **最终响应**：
    LLM 根据工具返回的结果生成最终的自然语言回答（“The current time is...”），NAT 将其打印到控制台。

<a id="run-server"></a>
## 2.2) Run as a NAT server

NAT provides another mechanism for running workflows through `nat serve`. `nat serve` creates and launches a REST FastAPI web server for interfacing with the toolkit as though it was an OpenAI-compatible endpoint. To learn more about all endpoints served by `nat serve`, please consult [this documentation](https://docs.nvidia.com/nemo/agent-toolkit/latest/reference/api-server-endpoints.html).

<span style="color: red"><i>note: If running this notebook in a cloud provider such as Google Colab, `dask` may be installed. If it is, you will first have to uninstall it via:</i></span>

In [ ]:
!uv pip uninstall dask

To start the FastAPI web server, issue the following command:

In [2]:
%%bash --bg
nat serve --config_file getting_started/configs/config.yml

It will take several seconds for the server to be reachable. The default port for the server is `8000` with `localhost` access.

Note that `--input` was not required for `nat serve`. To issue a request to the server, you can then do:

In [3]:
%%bash

# Issue a request to the background service
curl --request POST \
  --url http://localhost:8000/chat \
  --header 'Content-Type: application/json' \
  --data '{
    "messages": [
        {
          "role": "user",
          "content": "What is the current time?"
        }
      ]
    }' | jq

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   495  100   367  100   128     91     32  0:00:04  0:00:03  0:00:01   123  100   128     91     32  0:00:04  0:00:03  0:00:01   123


{
  "id": "960acaaa-99fa-4fe9-811f-2ef6105c72a9",
  "object": "chat.completion",
  "model": "unknown-model",
  "created": 1766021399,
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "The current time is 2025-12-18 01:29:55 +0000",
        "role": "assistant"
      }
    }
  ],
  "usage": {
    "prompt_tokens": 5,
    "completion_tokens": 7,
    "total_tokens": 12
  },
  "system_fingerprint": null,
  "service_tier": null
}


In [ ]:
# Terminate the process after completion
!pkill -9 -f "nat serve"

<a id="run-embedded"></a>
## 2.3) Running NAT Embedded within Python

The final way to run a NAT workflow is by embedding it into an already existing Python application or library.

Consider the following code:

In [ ]:
%%writefile nat_embedded.py
import asyncio
import sys

from nat.runtime.loader import load_config
from nat.utils import run_workflow


async def amain():
    config = load_config(sys.argv[1])
    query_num = 1
    try:
        while True:
            query = input()
            result = await run_workflow(config=config, prompt=query)
            print(f"Query {query_num}: {query}")
            print(f"Result {query_num}: {result}")
            query_num += 1
    except EOFError:
        pass


asyncio.run(amain())

Then we can run it as a normal Python program as shown below, or better yet, integrate with your existing services.

In [6]:
%%bash
python nat_embedded.py getting_started/configs/config.yml <<EOF
What are you capable of doing?
What does the 'current_datetime' tool do?
What does the 'getting_started' tool do?
What is the current time?
Can you echo back my name, Evan?
What is the current time?
Can you echo back my name, Will?
EOF

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Query 1: What are you capable of doing?
Result 1: I can use the following tools: current_datetime, getting_started
Query 2: What does the 'current_datetime' tool do?
Result 2: The 'current_datetime' tool returns the current date and time in human readable format with timezone information.
Query 3: What does the 'getting_started' tool do?
Result 3: The 'getting_started' tool takes a text input and echoes back with a pre-defined prefix.
Query 4: What is the current time?
Result 4: The current time is 2025-12-18 01:31:59 +0000.
Query 5: Can you echo back my name, Evan?
Result 5: Hello: Evan
Query 6: What is the current time?
Result 6: The current time is 2025-12-18 01:32:08 +0000
Query 7: Can you echo back my name, Will?
Result 7: Hello: Will
